# adapteddlo_muj — Quickstart Notebook

This notebook introduces the **Adapted Discrete Elastic Rod (DER) model for MuJoCo** from the
[RMDLO/adapteddlo_muj](https://github.com/RMDLO/adapteddlo_muj) repository (Bretl Group, UIUC).

It reproduces the **Localized Helical Buckling (LHB)** validation result from the paper using the
pre-computed simulation data already in this repository — no simulation run required.

**What you will do:**
1. Load pre-computed DER simulation snapshots for six discretisation levels (N = 40 … 180)
2. Compare the simulated rod shape to the analytical LHB solution
3. Measure convergence (RMS shape error vs N)
4. Learn the correct API for running live simulations

**Dependencies for Sections 1–3 (data only):** `numpy`, `matplotlib`, `pickle` (all standard)

**Dependencies for Section 4 (live simulation):**
`mujoco`, `gymnasium`, and the compiled `_Dlo_iso` C++ extension. See the build instructions below.

## 0. Setup

The data-only sections work with any Python 3.8+ environment. To also run live simulations,
build the DER C++ extension once from the repo root:

```bash
# 1. Install Python deps
pip install mujoco gymnasium numpy matplotlib

# 2. Build _Dlo_iso shared library
cd adapteddlo_muj/controllers/dlo_cpp
NUMPY_INC=$(python3 -c "import numpy; print(numpy.get_include())")
PYTHON_INC=$(python3 -c "from sysconfig import get_paths; print(get_paths()['include'])")
EIGEN_INC=/path/to/eigen   # e.g. ~/eigen, /usr/include/eigen3,
                           # or <mujoco_build>/_deps/eigen3-src
g++ -c Dlo_iso.cpp Dlo_iso_wrap.cpp Dlo_utils.cpp \
    -I"$EIGEN_INC" -I"$NUMPY_INC" -I"$PYTHON_INC" -fPIC -std=c++14 -O2
g++ -shared Dlo_iso.o Dlo_iso_wrap.o Dlo_utils.o -o _Dlo_iso.so -fPIC
```

> **Tip:** `Dlo_iso_wrap.cpp` (the SWIG-generated wrapper) is already committed to the repo,
> so `swig` is not needed. Eigen can also be installed via
> `conda install -c conda-forge eigen` or `apt install libeigen3-dev`.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pickle, os

print('numpy       ', np.__version__)
print('matplotlib  ', matplotlib.__version__)
print('All data-section dependencies found.')

## 1. Background: Discrete Elastic Rod (DER) in MuJoCo

The **Discrete Elastic Rod** model (Bergou et al., 2008) represents a slender elastic rod as a
sequence of rigid links connected by ball joints. Bending and torsional stiffness are enforced via
generalised torques applied each simulation step — not via passive joint springs.

`adapteddlo_muj` implements this for MuJoCo 3.x. The two key parameters are:

| Parameter | Physical meaning | Typical range |
|---|---|---|
| `alpha_bar` | Normalised **bending stiffness** (∝ EI) | 0.01 – 5.0 |
| `beta_bar`  | Normalised **torsional stiffness** (∝ GJ) | 0.01 – 2.0 |

Higher `alpha_bar` → stiffer rod (resists bending).  
Higher `beta_bar` → stiffer rod (resists twisting).

The model is validated against two analytical benchmarks:  
- **LHB** — Localized Helical Buckling (this notebook)  
- **MBI** — Michell's Buckling Instability (`scripts/dlo_testdata.py`)

## 2. Localized Helical Buckling (LHB) — Theory

A rod under end-twist and axial compression develops a **localized helical buckle**: a coiled region
in the middle while the rest remains nearly straight.

The analytical solution (Champneys & Thompson, 1996) predicts the deviation angle φ(s) as:

$$f(\varphi) = \frac{\cos\varphi - \cos\varphi_0}{1 - \cos\varphi_0} = \tanh^2\!\left(\frac{s}{s^*}\right)$$

where s\* is a characteristic length set by applied twist and rod geometry.

The pre-computed data in `adapteddlo_muj/data/lhb/adapt/` stores `[f(φ), s/s*]` from DER simulations
for N = 40, 60, 80, 110, 140, 180 segments.

## 3. Load Pre-Computed LHB Data

In [ ]:
# Locate the data directory relative to this notebook
REPO_ROOT = os.path.dirname(os.getcwd())   # one level up from notebooks/
DATA_DIR  = os.path.join(REPO_ROOT, 'adapteddlo_muj', 'data', 'lhb', 'adapt')

N_list   = [40, 60, 80, 110, 140, 180]
lhb_data = {}

for N in N_list:
    with open(os.path.join(DATA_DIR, f'lhb{N}.pickle'), 'rb') as f:
        d = pickle.load(f)
    fphi_sim = np.array(d[0])   # f(φ): shape function from DER sim
    s_ss_sim = np.array(d[1])   # s/s*: normalised arc length
    lhb_data[N] = (fphi_sim, s_ss_sim)
    print(f'N={N:3d}: {len(fphi_sim)} nodes  '
          f'  s/s* ∈ [{s_ss_sim.min():.2f}, {s_ss_sim.max():.2f}]  '
          f'  f(φ) ∈ [{fphi_sim.min():.2f}, {fphi_sim.max():.2f}]')

## 4. DER Simulation vs Analytical Solution

In [ ]:
colors   = plt.cm.plasma(np.linspace(0.1, 0.85, len(N_list)))
s_ref    = np.linspace(-6, 6, 500)
fphi_ref = np.tanh(s_ref)**2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: shape comparison ──────────────────────────────────────────────────
ax = axes[0]
ax.plot(s_ref, fphi_ref, 'k-', lw=2.5, zorder=10, label='Analytical: tanh²(s/s*)')

rms_errors = []
for N, c in zip(N_list, colors):
    fphi_sim, s_ss_sim = lhb_data[N]
    mask = np.abs(s_ss_sim) <= 6          # keep the well-sampled central region
    ax.plot(s_ss_sim[mask], fphi_sim[mask], '-o', color=c,
            lw=1.5, ms=2, label=f'DER N={N}')
    rms = float(np.sqrt(np.mean(
        (fphi_sim[mask] - np.tanh(s_ss_sim[mask])**2)**2
    )))
    rms_errors.append(rms)
    print(f'N={N:3d}  RMS deviation = {rms:.4f}')

ax.set_xlabel('s / s*  (normalised arc length)', fontsize=11)
ax.set_ylabel('f(φ) = (cos φ − cos φ₀) / (1 − cos φ₀)', fontsize=10)
ax.set_title('Localized Helical Buckling (LHB)\nDER simulation vs analytical solution', fontsize=11)
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(-7, 7)
ax.set_ylim(-0.05, 1.1)

# ── Right: convergence ──────────────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(N_list, rms_errors, 'b-o', ms=7, lw=2)
for x, y in zip(N_list, rms_errors):
    ax2.annotate(f'{y:.3f}', (x, y),
                 textcoords='offset points', xytext=(0, 8),
                 fontsize=8, ha='center')
ax2.set_xlabel('Number of segments N', fontsize=11)
ax2.set_ylabel('RMS deviation from tanh²(s/s*)', fontsize=11)
ax2.set_title('DER convergence with discretisation\n(LHB validation, adapted controller)', fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lhb_quickstart.pdf', bbox_inches='tight')
plt.show()
print(f'\nFigure saved to lhb_quickstart.pdf')

### Interpretation

Expected results:

| N | RMS deviation |
|---:|---:|
| 40  | 0.157 |
| 60  | 0.096 |
| 80  | 0.063 |
| 110 | 0.039 |
| 140 | 0.027 |
| 180 | 0.019 |

RMS error decreases monotonically with N, confirming that the DER simulation converges to the
continuous analytical solution.

**Rule of thumb:** For DLO manipulation, N ≈ 30–60 segments per metre of cable balances
accuracy and real-time simulation speed.

## 5. Running a Live Simulation

This section shows the **correct API pattern** for running the DER controller in MuJoCo.
It requires the compiled `_Dlo_iso.so` (see Setup above) and `mujoco` installed.

The per-step loop is:
1. `dlo.update_torque()` — compute DER generalised forces and write to `data.qfrc_passive`
2. `sim.step()` — advance MuJoCo by one timestep
3. `sim.forward()` — recompute forward kinematics

The LHB-validated configuration (`r_len=9.29`, `alpha_bar=1.345`, `beta_bar=0.789`) is the
geometry used in the paper and is the recommended starting point.

In [ ]:
# ── Requires: mujoco, gymnasium, compiled _Dlo_iso.so ──────────────────────
# Complete the build step in Section 0 before running this cell.

import sys, os
import numpy as np
import mujoco

# Make _Dlo_iso.so importable
import adapteddlo_muj as _pkg
_dlo_cpp = os.path.join(os.path.dirname(_pkg.__file__), 'controllers', 'dlo_cpp')
if _dlo_cpp not in sys.path:
    sys.path.insert(0, _dlo_cpp)

from adapteddlo_muj.assets.genrope.gdv_O_weld2 import GenKin_O_weld2
from adapteddlo_muj.utils.xml_utils import XMLWrapper
from adapteddlo_muj.utils.mjc_utils import MjSimWrapper
import adapteddlo_muj.utils.mjc2_utils as mjc2
from adapteddlo_muj.controllers.ropekin_controller_adapt import DLORopeAdapt

# ── Parameters ─────────────────────────────────────────────────────────────
r_len       = 9.29    # rod arc length [m]
r_pieces    = 40      # number of DER segments
r_thickness = 0.04    # capsule radius [m]
alpha_bar   = 1.345   # normalised bending stiffness
beta_bar    = 0.789   # normalised torsional stiffness

# ── Build MuJoCo XML ───────────────────────────────────────────────────────
asset_dir  = os.path.join(os.path.dirname(_pkg.__file__), 'assets')
world_path = os.path.join(asset_dir, 'world_test.xml')
box_path   = os.path.join(asset_dir, 'anchorbox.xml')
rope_path  = os.path.join(asset_dir, 'dlorope1dkin.xml')

# GenKin_O_weld2 writes both the rope XML and a matching anchorbox XML.
# The anchor bodies are placed at ±r_len/2 along x to match the rope endpoints.
GenKin_O_weld2(
    r_len=r_len, r_thickness=r_thickness, r_pieces=r_pieces,
    j_stiff=0.0, j_damp=1.0,
    init_pos=[r_len / 2, 0.0, 0.5], init_quat=[1., 0., 0., 0.],
    d_small=0., rope_type='capsule', vis_subcyl=False,
    obj_path=rope_path,
)

xml_world = XMLWrapper(world_path)
xml_world.merge_multiple(XMLWrapper(box_path),  ['worldbody', 'equality', 'contact'])
xml_world.merge_multiple(XMLWrapper(rope_path), ['worldbody'])
xml_str = xml_world.get_xml_string()

# ── Build model and DER controller ─────────────────────────────────────────
model = mujoco.MjModel.from_xml_string(xml_str)
data  = mujoco.MjData(model)
sim   = MjSimWrapper(model, data)

model.opt.gravity[:] = 0.   # gravity-free for elastic mechanics validation

# sim.forward() MUST be called before DLORopeAdapt so that site positions
# are populated when DLORopeAdapt computes rest edge lengths (e_bar) in __init__.
sim.forward()

dlo = DLORopeAdapt(
    model=model, data=data,
    n_link=r_pieces,
    alpha_bar=alpha_bar, beta_bar=beta_bar,
    overall_rot=0.,
    bothweld=True,   # both ends clamped — required for LHB / torsion tests
)

# ── Step loop ──────────────────────────────────────────────────────────────
N_STEPS  = 500
site_ids = [mjc2.obj_name2id(model, 'site', f'S_{i}') for i in range(r_pieces)]
site_ids.append(mjc2.obj_name2id(model, 'site', 'S_last'))

for _ in range(N_STEPS):
    dlo.update_torque()   # DER elastic forces → data.qfrc_passive
    sim.step()            # MuJoCo time integration
    sim.forward()         # forward kinematics update

node_pos = np.array([data.site_xpos[s].copy() for s in site_ids])  # (r_pieces+1, 3)
print(f'Ran {N_STEPS} steps. Node position array shape: {node_pos.shape}')
print(f'Rod extents — x: [{node_pos[:,0].min():.3f}, {node_pos[:,0].max():.3f}] m')
print(f'              z: [{node_pos[:,2].min():.3f}, {node_pos[:,2].max():.3f}] m')

### Changing stiffness at runtime

You can update `alpha_bar` and `beta_bar` without rebuilding the model:

```python
dlo.change_ropestiffness(alpha_bar=0.5, beta_bar=0.3)
```

This is useful for **domain randomisation** in RL training — randomise material stiffness
at each episode reset to build policies that transfer to real cables with uncertain properties.

## 6. Next Steps

| Goal | Where to look |
|---|---|
| Run the full LHB / MBI validation tests | `scripts/dlo_testdata.py` |
| Identify ᾱ, β̄ from a real cable video | `scripts/real2sim_paramiden.py` |
| Gymnasium environments for RL | `adapteddlo_muj/envs/` |
| Wire plugin variant (faster, requires build) | `scripts/plugin_test/` |
| Sim-to-real shape comparison plots | `scripts/simvreal_dlomuj.py` |

Found a bug or want to add a feature? Open an issue or PR on
[github.com/RMDLO/adapteddlo_muj](https://github.com/RMDLO/adapteddlo_muj).